# Longrun Z300-EP100 Source Maps

### 目的
把 Hindcast 05 中的 Z300 pointwise correlation map 拓展到 longrun 数据上：把每个模式年当作一个样本，计算 Jan、Feb、Mar 的 Z300 stationary anomaly 与 EP100 wave activity 的 year-to-year correlation。

### 科学问题
在 INT-3D、CLIM-3D、CLIM-2D longrun 中，300 hPa 对流层 stationary-wave pattern 的年际变化，是否对应 100 hPa、40-80N 的 upward EP flux 分量变化？这种关系是 all-wave、wave1、wave2、wave1+wave2 还是 rest/synoptic waves 更明显？

### 方法
相关分析需要逐年样本：Z300 从 longrun Z3 原始 hybrid-level 文件插值到 300 hPa，取月平均并去除 zonal mean 得到每个模式年的 stationary anomaly。EP100 定义为 `mean(-ep2)`，100 hPa，40-80N，正值表示更强 upward wave activity，不是 divergence。每个 case-month 在所有可用模式年上做 pointwise Pearson correlation。黑色等值线优先直接读取 `climatology/Z3_climatology_plev_doy.nc` 的多年平均 Z300 stationary wave；只有该文件缺失时才回退到逐年 Z300 样本平均。图上不包含 O3 RMSE。

### 预期输出
输出目录为 `Longrun/Visualization/plots/longrun_z300_ep100_source_maps/`，包含 `figures/`、`tables/`、`cache/`。每张图同时保存 png/pdf，每张图有 summary csv。

### 解读
填色为 Z300 gridpoint correlation r；黑色等值线为同一 experiment、同一月份的 climatological stationary waves。若相关中心贴近 climatological ridge/trough，支持 constructive/destructive interference 作为 EP100 variability source 的解释。

### 注意
默认跑完整样本。调试时可设置环境变量 `LONGRUN_Z300_EP100_MAX_YEARS=3`，输出会自动加 `_quick3` 后缀，避免覆盖正式结果。注意：`Climatology.ipynb` 生成的是多年平均气候态，没有 year 维度，所以不能替代这里用于相关计算的逐年 Z300 样本。


### 导入、路径和实验定义

### 目的
定义 longrun 三组实验、输出目录、月份窗口和画图依赖。

### 科学问题
这一步不直接回答科学问题，但保证 CLIM-3D、CLIM-2D、INT-3D 使用同一套变量定义和绘图风格。

### 方法
复用 `Extention_analysis/hindcast_extension_utils.py` 中的压力插值、纬度加权平均、日期转换等函数。输出只写入 `Longrun/Visualization/plots/longrun_z300_ep100_source_maps/`。

### 预期输出
打印输出根目录、是否 quick mode、每个实验的数据目录。

### 解读
如果路径存在且 quick mode 正确，后续代码可以从 longrun 数据目录构建缓存。

### 注意
不会修改 `/mnt/soclim0/public_data/weiji` 下的原始数据。


In [ ]:
from pathlib import Path
import os
import re
import sys
import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-code-cleaned")

CODE_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
UTIL_ROOT = CODE_ROOT / "Hindcast_experiment" / "Extention_analysis"
if str(UTIL_ROOT) not in sys.path:
    sys.path.insert(0, str(UTIL_ROOT))

from hindcast_extension_utils import (
    _interp_profile_logp,
    coslat_weighted_mean,
    date_to_doy,
    mmdd_to_doy,
    LAT_EP,
    LAT_Z300,
)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
WORK_ROOT = CODE_ROOT / "Longrun" / "Visualization"
OUT_ROOT = WORK_ROOT / "plots" / "longrun_z300_ep100_source_maps"
FIG_DIR = OUT_ROOT / "figures"
TAB_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TAB_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CASE_SPECS = {
    "INT-3D": {
        "root": DATA_ROOT / "B2000WCN001002_timefixed",
        "title": "INT-3D",
    },
    "CLIM-3D": {
        "root": DATA_ROOT / "B2000WCN007009010011_Clim3D_timefixed",
        "title": "CLIM-3D",
    },
    "CLIM-2D": {
        "root": DATA_ROOT / "B2000WCN_NOCOUPL001002_timefixed",
        "title": "CLIM-2D",
    },
}

MONTH_WINDOWS = {
    "Jan": ((1, 1), (1, 31)),
    "Feb": ((2, 1), (2, 28)),
    "Mar": ((3, 1), (3, 31)),
}
WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
EP_PLEV_HPA = 100.0
Z_PLEV_HPA = 300.0
MAX_YEARS_ENV = int(os.environ.get("LONGRUN_Z300_EP100_MAX_YEARS", "0") or 0)
MAX_YEARS = MAX_YEARS_ENV if MAX_YEARS_ENV > 0 else None
RUN_SUFFIX = f"_quick{MAX_YEARS}" if MAX_YEARS else ""

print("OUT_ROOT", OUT_ROOT)
print("MAX_YEARS", MAX_YEARS)
for label, spec in CASE_SPECS.items():
    print(label, spec["root"], "exists=", spec["root"].exists())


### 地图绘图函数

### 目的
定义与 Hindcast TEST_TROPOS / Extention_analysis 05 一致的极区地图绘图函数。

### 科学问题
统一投影和比例，避免因为地图投影差异误读 Z300 correlation pattern。

### 方法
若 cartopy 可用，使用 `NorthPolarStereo` 子图投影、`PlateCarree` 数据投影、20-90N extent、coastline/borders/gridlines；longitude 转换到 -180..180 并添加 cyclic point，避免接缝。

### 预期输出
本代码块只定义函数，不直接保存图。

### 解读
后续 longrun map 与 hindcast 05 的视觉设置一致，可以直接比较相关中心和 climatological stationary wave contours。

### 注意
如果 cartopy 不可用，会回退到普通 lat-lon 图，并在 summary 中记录。


In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.util import add_cyclic_point
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False
    ccrs = None
    cfeature = None
    add_cyclic_point = None


def map_lon_lat_values(da: xr.DataArray):
    lon = np.asarray(da["lon"].values, dtype=float)
    lat = np.asarray(da["lat"].values, dtype=float)
    values = np.asarray(da.transpose("lat", "lon").values, dtype=float)
    if lat[0] > lat[-1]:
        lat = lat[::-1]
        values = values[::-1, :]
    lon_wrapped = ((lon + 180.0) % 360.0) - 180.0
    order = np.argsort(lon_wrapped)
    lon_sorted = lon_wrapped[order]
    values_sorted = values[:, order]
    if add_cyclic_point is None:
        return lon_sorted, lat, values_sorted
    try:
        values_cyc, lon_cyc = add_cyclic_point(values_sorted, coord=lon_sorted)
        return lon_cyc, lat, values_cyc
    except ValueError:
        return lon_sorted, lat, values_sorted


def add_polar_map_features(ax, data_crs):
    try:
        ax.add_feature(cfeature.COASTLINE.with_scale("110m"), linewidth=0.55, edgecolor="0.25")
        ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=0.28, edgecolor="0.35", alpha=0.75)
        ax.gridlines(crs=data_crs, linewidth=0.32, color="0.45", alpha=0.45, linestyle="--", draw_labels=False)
    except Exception as exc:
        ax.text(0.02, 0.02, f"map features unavailable: {type(exc).__name__}", transform=ax.transAxes, fontsize=7, ha="left", va="bottom", color="0.35")


def draw_corr_panel(ax, corr_da, stationary_da, title, vlim=0.8, wave_levels=np.arange(-240, 241, 40)):
    lon, lat, corr_values = map_lon_lat_values(corr_da)
    _, _, wave_values = map_lon_lat_values(stationary_da)
    levels = np.linspace(-vlim, vlim, 17)
    if HAS_CARTOPY:
        data_crs = ccrs.PlateCarree()
        im = ax.contourf(lon, lat, corr_values, levels=levels, cmap="RdBu_r", extend="both", transform=data_crs)
        ax.contour(lon, lat, wave_values, levels=wave_levels, colors="k", linewidths=0.62, alpha=0.84, transform=data_crs)
        ax.set_extent([-180.0, 180.0, LAT_Z300[0], LAT_Z300[1]], crs=data_crs)
        add_polar_map_features(ax, data_crs)
    else:
        im = ax.contourf(lon, lat, corr_values, levels=levels, cmap="RdBu_r", extend="both")
        ax.contour(lon, lat, wave_values, levels=wave_levels, colors="k", linewidths=0.62, alpha=0.84)
        ax.set_xlim(-180, 180)
        ax.set_ylim(*LAT_Z300)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    ax.set_title(title, fontsize=8)
    return im


### 数据构建函数：EP100 和 Z300 年际样本

### 目的
构建每个 longrun experiment、每个月份的 EP100 年际指标和 Z300 年际 stationary anomaly 场。

### 科学问题
为了回答 Z300 是否解释 EP100，需要把每个模式年变成一个样本，并保证 Z300 与 EP100 使用同一个月份和同一组年份。

### 方法
EP100 从 `EPflux_daily_ubar_wcorr/<wave>/EPFLUX_*_time_plev_lat.nc` 读取，取 `-ep2`、100 hPa、40-80N cos-lat mean、月平均。用于相关分析的 Z300 仍需逐年样本，因此从每年的 Z3 文件读取，对 Jan/Feb/Mar 时间段插值到 300 hPa、月平均、20-90N。Z300 cache 写入 `cache/<case>_Z300_<month>.nc`。用于黑色等值线的 climatological stationary wave 优先读取 longrun 已有 `climatology/Z3_climatology_plev_doy.nc`。

### 预期输出
本代码块定义缓存和计算函数；执行下一代码块时会写出 EP100/Z300 cache 和 summary tables。

### 解读
`MAX_YEARS` 只用于调试；正式运行应为 `None`，使用全部 210/216/205 年。

### 注意
Z300 插值是最耗时步骤；已有逐年 Z300 cache 时会直接读取。`Z3_climatology_plev_doy.nc` 只用于背景等值线，不能用于 year-to-year correlation。


In [ ]:
def month_doy_bounds(month):
    start, end = MONTH_WINDOWS[month]
    return mmdd_to_doy(*start), mmdd_to_doy(*end)


def z3_year_from_file(path: Path) -> int:
    m = re.search(r"\.(\d{4})\.Z3\.nc$", path.name)
    if not m:
        raise ValueError(f"cannot parse Z3 year from {path.name}")
    return int(m.group(1))


def sorted_z3_files(root: Path):
    files = sorted((root / "Z3").glob("*.Z3.nc"), key=z3_year_from_file)
    if MAX_YEARS:
        files = files[:MAX_YEARS]
    return files


def epflux_file(root: Path, wave: str) -> Path:
    matches = sorted((root / "EPflux_daily_ubar_wcorr" / wave).glob(f"EPFLUX_{wave}_*yr_time_plev_lat.nc"))
    if not matches:
        raise FileNotFoundError(f"missing EPFlux {wave} below {root}")
    return matches[0]


def load_ep100_monthly(root: Path, month: str) -> pd.DataFrame:
    start_doy, end_doy = month_doy_bounds(month)
    frames = []
    for wave in WAVES:
        path = epflux_file(root, wave)
        with xr.open_dataset(path, decode_times=False) as ds:
            time = np.floor(np.asarray(ds["time"].values, dtype=float)).astype(int)
            year = time // 365 + 1
            doy = time % 365 + 1
            mask = (doy >= start_doy) & (doy <= end_doy)
            if MAX_YEARS:
                mask = mask & (year <= MAX_YEARS)
            da = -ds["ep2"].sel(plev=EP_PLEV_HPA * 100.0, method="nearest").isel(time=mask)
            metric = coslat_weighted_mean(da, LAT_EP)
            metric = metric.assign_coords(year=("time", year[mask])).groupby("year").mean("time", skipna=True).load()
        frames.append(pd.DataFrame({"year": metric["year"].values.astype(int), f"EP100_{wave}": metric.values.astype(float)}))
    out = frames[0]
    for frame in frames[1:]:
        out = out.merge(frame, on="year", how="inner")
    out["EP100_wave1_plus_wave2"] = out["EP100_wave1"] + out["EP100_wave2"]
    out["month"] = month
    return out


def build_z300_monthly(case_key: str, root: Path, month: str) -> xr.DataArray:
    cache = CACHE_DIR / f"{case_key}_Z300_{month}{RUN_SUFFIX}.nc"
    if cache.exists():
        return xr.open_dataarray(cache).load()
    start_doy, end_doy = month_doy_bounds(month)
    fields = []
    years = []
    files = sorted_z3_files(root)
    for i, path in enumerate(files, start=1):
        year = z3_year_from_file(path)
        with xr.open_dataset(path, decode_times=False) as ds:
            doy = date_to_doy(np.asarray(ds["date"].values, dtype=np.int64))
            mask = (doy >= start_doy) & (doy <= end_doy)
            if mask.sum() == 0:
                continue
            sub = ds.isel(time=mask).sel(lat=slice(LAT_Z300[0], LAT_Z300[1]))
            p_mid = sub["hyam"] * sub["P0"] + sub["hybm"] * sub["PS"]
            z300 = _interp_profile_logp(
                sub["Z3"].transpose("time", "lat", "lon", "lev"),
                p_mid.transpose("time", "lat", "lon", "lev"),
                Z_PLEV_HPA * 100.0,
            ).mean("time", skipna=True).load()
        fields.append(z300.astype(np.float32))
        years.append(year)
        if i % 25 == 0:
            print(f"{case_key} {month}: processed {i}/{len(files)} Z3 years")
    if not fields:
        raise RuntimeError(f"no Z300 fields built for {case_key} {month}")
    da = xr.concat(fields, dim=pd.Index(years, name="year"))
    da.name = "Z300"
    da.attrs.update({"units": "m", "plev_hpa": Z_PLEV_HPA, "month": month, "case": case_key})
    da.to_netcdf(cache)
    return da


def stationary_anomaly(z300: xr.DataArray) -> xr.DataArray:
    return z300 - z300.mean("lon", skipna=True)


def load_z300_stationary_climatology(case_key: str, root: Path, month: str, fallback_z300: xr.DataArray | None = None):
    """Load climatological stationary Z300 contours from Climatology.ipynb output.

    The longrun correlation maps still need year-by-year Z300 fields. This
    helper is only for the black climatological stationary-wave contours.
    """
    clim_path = root / "climatology" / "Z3_climatology_plev_doy.nc"
    start_doy, end_doy = month_doy_bounds(month)
    if clim_path.exists():
        with xr.open_dataset(clim_path, decode_times=False) as ds:
            if "Z3_clim_all" not in ds:
                raise KeyError(f"Z3_clim_all missing in {clim_path}")
            da = (
                ds["Z3_clim_all"]
                .sel(plev=Z_PLEV_HPA * 100.0, method="nearest")
                .sel(doy=slice(start_doy, end_doy))
                .sel(lat=slice(LAT_Z300[0], LAT_Z300[1]))
                .mean("doy", skipna=True)
                .load()
            )
        da = stationary_anomaly(da)
        da.name = "Z300_stationary_climatology"
        da.attrs.update({
            "units": "m",
            "plev_hpa": Z_PLEV_HPA,
            "month": month,
            "case": case_key,
            "source": str(clim_path),
        })
        return da.astype(np.float32), "Z3_climatology_plev_doy.nc"
    if fallback_z300 is None:
        raise FileNotFoundError(f"missing {clim_path} and no fallback Z300 sample supplied")
    return stationary_anomaly(fallback_z300).mean("year", skipna=True), "fallback_mean_of_yearly_Z300_cache"


def pointwise_year_corr(field_year_lat_lon: xr.DataArray, metric: pd.Series) -> xr.DataArray:
    metric = metric.dropna()
    years = np.intersect1d(field_year_lat_lon["year"].values.astype(int), metric.index.astype(int))
    if len(years) < 3:
        raise ValueError("need at least three shared years for correlation")
    field = field_year_lat_lon.sel(year=years)
    metric_da = xr.DataArray(metric.loc[years].values.astype(float), coords={"year": years}, dims="year")
    return xr.corr(field, metric_da, dim="year")


### 绘制 longrun Z300-EP100 pointwise maps

### 目的
为 INT-3D、CLIM-3D、CLIM-2D 的 Jan、Feb、Mar 分别绘制 Z300 vs EP100 分波 correlation maps。

### 科学问题
不同 longrun experiment 中，哪一类 EP100 分波最能由对流层 Z300 stationary-wave pattern 的年际变化解释？

### 方法
每张图包含五个 EP100 target：all waves、wave1、wave2、wave1+wave2、wave rest。填色为跨年份 pointwise correlation；黑色等值线是从该 experiment 的 `climatology/Z3_climatology_plev_doy.nc` 读取并取同月平均的 climatological stationary Z300。所有地图使用与 hindcast 05 一致的 NorthPolarStereo 设置。

### 预期输出
`figures/Longrun_Z300_EP100_pointwise_corr_<case>_<month>.png/pdf`，以及 `tables/Longrun_Z300_EP100_pointwise_corr_summary.csv` 和每张图的 summary csv。

### 解读
如果 W1/W2/W1+W2 panel 的相关中心明显贴近 climatological stationary wave contours，说明该 experiment 的 EP100 variability 可能受对流层 planetary-wave geometry 控制。

### 注意
这张图只做 dynamics source diagnosis，不包含 O3 RMSE 或臭氧结果。


In [ ]:
TARGETS = [
    ("EP100_all_waves", "All waves"),
    ("EP100_wave1", "Wave 1"),
    ("EP100_wave2", "Wave 2"),
    ("EP100_wave1_plus_wave2", "Wave 1 + Wave 2"),
    ("EP100_wave_rest", "Rest / synoptic waves"),
]

summary_rows = []
for case_key, spec in CASE_SPECS.items():
    root = spec["root"]
    for month in MONTH_WINDOWS:
        print(f"Building {case_key} {month}")
        ep = load_ep100_monthly(root, month)
        z300 = build_z300_monthly(case_key.replace("-", "_"), root, month)
        za = stationary_anomaly(z300)
        stationary_target, contour_source = load_z300_stationary_climatology(case_key, root, month, fallback_z300=z300)
        common_years = np.intersect1d(za["year"].values.astype(int), ep["year"].values.astype(int))
        ep = ep.loc[ep["year"].isin(common_years)].set_index("year").sort_index()
        za = za.sel(year=common_years)
        subplot_kw = {"projection": ccrs.NorthPolarStereo()} if HAS_CARTOPY else {}
        fig, axes = plt.subplots(2, 3, figsize=(11.8, 8.4), subplot_kw=subplot_kw, constrained_layout=True)
        im = None
        case_rows = []
        for ax, (col, title) in zip(axes.ravel(), TARGETS):
            corr = pointwise_year_corr(za, ep[col])
            im = draw_corr_panel(ax, corr, stationary_target, title)
            row = {
                "case": case_key,
                "month": month,
                "target": col,
                "n_years": int(len(common_years)),
                "R_mean": float(corr.mean(skipna=True).values),
                "R_absmax": float(abs(corr).max(skipna=True).values),
                "projection": "NorthPolarStereo" if HAS_CARTOPY else "plain-latlon",
                "contour_source": contour_source,
                "quick_max_years": MAX_YEARS if MAX_YEARS else np.nan,
            }
            case_rows.append(row)
            summary_rows.append(row)
        for ax in axes.ravel()[len(TARGETS):]:
            ax.axis("off")
        if im is not None:
            cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.76, pad=0.02)
            cbar.set_label("Correlation r", fontsize=8)
            cbar.ax.tick_params(labelsize=8)
        fig.suptitle(f"{case_key} {month}: Z300 stationary anomaly vs EP100\nEP100 = mean(-ep2), 100 hPa, 40-80N; contours = {case_key} climatological stationary Z300", fontsize=10)
        stem = f"Longrun_Z300_EP100_pointwise_corr_{case_key.replace('-', '_')}_{month}{RUN_SUFFIX}"
        fig.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
        pd.DataFrame(case_rows).to_csv(TAB_DIR / f"{stem}_summary.csv", index=False)
        ep.reset_index().to_csv(TAB_DIR / f"Longrun_EP100_metrics_{case_key.replace('-', '_')}_{month}{RUN_SUFFIX}.csv", index=False)
        plt.close(fig)

summary = pd.DataFrame(summary_rows)
summary_csv = TAB_DIR / f"Longrun_Z300_EP100_pointwise_corr_summary{RUN_SUFFIX}.csv"
summary.to_csv(summary_csv, index=False)
print(summary.to_string(index=False))
print("Wrote", summary_csv)
